# Capstone A: the night shift

Run an overnight shift alone for an accounting firm in Brisbane: an unlabelled, noisy queue of twelve alerts, one real intrusion hiding in it, and a morning team waiting for your escalation, a new detection and a shift report.

The files for this seminar are in `~/case` (Linux) and `case/` (Python): `alerts-night.json`, `capture.pcap`, `dns.log`, `eventlogs/Security.jsonl`, `eventlogs/Sysmon.jsonl`, `eventlogs/System.jsonl`, `fw_sessions.csv`, `intel.json`, `process.csv`, `README.txt`.

## 1. How the night desk works

A capstone uses every program from seminars 2 and 3 at once, so start by seeing how they fit. Each program answers one kind of question, and a night shift moves through them in order:

| Program | The question it answers | Reads |
|---|---|---|
| Alert queue | What do I have to decide tonight? | alerts-night.json |
| SIEM search | How big is it: which hosts, how often, how much data? | fw_sessions.csv, dns.log, process.csv, the event logs |
| Event viewer | What exactly ran on the laptop, started by what? | eventlogs/*.jsonl |
| Packet viewer | What really crossed the wire, and to where? | capture.pcap |
| Indicators and ATT&CK | Has anyone seen this before, and what do we call it? | intel.json |
| Sigma rules | How do we catch it sooner next time? | the event logs |
| Report builder | What does the client need to decide, and what does the morning shift need to know? | your notes |

The client tonight is **Wattle & Finch Accountants** in Brisbane. Their working day is your night. Read `README.txt` first, then run the inventory search below: it shows which files the SIEM can search and how many events each holds. The alert queue and the intel feed are JSON, so the SIEM reads them too. The packet capture isn't in the SIEM; the packet viewer reads it.

- [ ] Run the inventory search
- [ ] Which file holds the most events? Which one is about the whole office, and which only about one laptop?
- [ ] Note which program you would open for each kind of question

> Hint: `stats count by source sourcetype` makes one row per file: eight files, from 308 events. The firewall and DNS logs cover every laptop; the event logs and process list come from one laptop only.

```tool siem
{"q": "* | stats count by source sourcetype", "n": 8, "files": ["fw_sessions.csv", "dns.log", "process.csv", "eventlogs/Sysmon.jsonl", "eventlogs/Security.jsonl", "eventlogs/System.jsonl"], "blank": ["q"]}
```

?> Which evidence file holds the most events? :: sha256:389616bb3f05f22f67c324c1da02da6bbb02f7792854469c24dfb3bab4474c03

## 2. Start of shift

You are the overnight analyst at a managed SOC. The queue has **twelve alerts and no severities**: the client's old SIEM doesn't send them. One real intrusion is in there. The rest is ordinary noise.

This is a capstone: the steps say what to deliver, not how. **Hints cost one point each** on the rubric, so open one only when you are stuck, and write down how many you opened.

Start by hashing the evidence and reading the queue end to end before you touch anything.

- [ ] Hash every evidence file and keep the list
- [ ] Read all twelve alerts once, quickly
- [ ] Write down your first guess of which ones belong together: you will test it

> Hint: Group alerts by host first. One laptop appears far more often than any other.

?> How many alerts are in the night queue? :: sha256:6b51d431df5d7f141cbececcf79edf3dd861c3b4069f0b11661a3eefacbba918

In [ ]:
%%bash
cd ~/case
sha256sum *.* eventlogs/*
echo ---
grep -c '"id"' alerts-night.json
grep -E '"(id|host|rule)"' alerts-night.json

## 3. Work the queue

Give every alert a disposition and a reason. A reason is a log line or a fact you checked, not a feeling. Severity is yours to judge: the queue doesn't tell you.

- [ ] Every alert has a disposition
- [ ] Every reason points at evidence
- [ ] Alerts that belong to one intrusion say so
- [ ] Use "needs more information" only with a question someone can answer

> Hint: Five alerts are on acc-lt03. A program from a three-day-old domain at 00:03, a remote access tool at 00:04 and a new program in ProgramData at 00:13 on the same laptop are not a coincidence.

```tool alerts
{"file": "alerts-night.json", "d": {"AL-8401": "escalate", "AL-8402": "benign", "AL-8403": "escalate", "AL-8404": "escalate", "AL-8405": "benign", "AL-8406": "escalate", "AL-8407": "benign", "AL-8408": "escalate", "AL-8409": "benign", "AL-8410": "benign", "AL-8411": "benign", "AL-8412": "more"}, "n": 5, "blank": ["d"]}
```

?> Which host do the related alerts share? :: sha256:7006306a1e36f893caf6a9147da511e6d6f864ecf30bcfd51d8b07cd29285ba5

## 4. Scope it with search

Before you escalate, know how big it is. Which laptop sent the most data out, where did it go, did anyone else talk to the same places, and what did the firewall stop?

- [ ] Which host sent the most bytes out? Is that normal for it?
- [ ] Who sends data to storage.cloudbox.example, and how much? (check intel.json: is that site allowed?)
- [ ] Who else asked DNS for ledgerpro-support.example?
- [ ] What did the firewall deny?
- [ ] Write the cloud-storage search in KQL

> Hint: `stats sum(bytes_out) as sent by src_host` adds up the bytes per laptop. Reception sends files to clients through cloud storage every day; compare its numbers with acc-lt03's.

```tool siem
{"q": "source=*fw_sessions.csv | stats sum(bytes_out) as sent by src_host | sort - sent", "n": 6, "files": ["fw_sessions.csv"], "blank": ["q"]}
```

```tool siem
{"q": "source=*fw_sessions.csv dst_host=storage.cloudbox.example | stats count sum(bytes_out) as sent by src_host", "n": 2, "files": ["fw_sessions.csv"], "blank": ["q"]}
```

```tool siem
{"q": "source=*dns.log ledgerpro-support.example | stats count by src_ip", "n": 1, "files": ["dns.log"], "blank": ["q"]}
```

```tool siem
{"q": "source=*fw_sessions.csv action=deny | table _time src_host dst_host dport app", "n": 2, "files": ["fw_sessions.csv"], "blank": ["q"]}
```

```tool siem
{"q": "fw_sessions | where dst_host == \"storage.cloudbox.example\" | summarize sum(bytes_out) by src_host", "lang": "kql", "n": 2, "files": ["fw_sessions.csv"], "blank": ["q"]}
```

?> Which host sent the most data out? :: sha256:7006306a1e36f893caf6a9147da511e6d6f864ecf30bcfd51d8b07cd29285ba5
?> How many hosts sent data to storage.cloudbox.example? :: sha256:d4735e3a265e16eee03f59718b9b5d03019c07d8b6c51f90da3a666eec13ab35
?> Which site did the firewall block for acc-lt03? :: sha256:4907c3569357f30a966c09a74b595259cb8cff54b2994185944fa02f30aef176

## 5. Confirm on the host and the wire

Search results are a summary. Confirm on the laptop's own logs and on the packet capture: which programs ran, which one sent the data, what it really is, and where the data went.

- [ ] Sysmon event 1: follow the chain from the download to the copy tool
- [ ] Sysmon event 1 shows OriginalFileName, the name the program was built with. Which program is hiding under a new name?
- [ ] Sysmon event 3: which program connected to 203.0.113.150, and how many times?
- [ ] Capture: find the TLS Client Hellos to the storage site

> Hint: In the packet viewer, the filter `tls.handshake.extensions_server_name contains cloudbox` shows the start of each upload stream. rclone is a real, free copy tool for cloud storage: useful to IT, and to anyone stealing files.

```tool events
{"file": "eventlogs/Sysmon.jsonl", "ids": "1", "n": 17, "blank": ["ids", "src", "q"]}
```

```tool events
{"file": "eventlogs/Sysmon.jsonl", "ids": "1", "q": "rclone.exe", "n": 3, "blank": ["ids", "src", "q"]}
```

```tool events
{"file": "eventlogs/Sysmon.jsonl", "ids": "3", "q": "203.0.113.150", "n": 8, "blank": ["ids", "src", "q"]}
```

```tool packets
{"file": "capture.pcap", "f": "tls.handshake.extensions_server_name contains cloudbox", "n": 8, "blank": ["f"]}
```

?> Which domain did the remote access tool connect to? :: sha256:30a80820053d3b0b7c245df61b45c1cca4c778fea876a09584283525046c57c0
?> What file name does the copy tool run under? :: sha256:ffc53d6ac58004e850e60bf30ab4c7fee651a25fd28be01932ca8959f9cb37a0
?> What is its real program name (OriginalFileName)? :: sha256:d5604ae4cfd963fe62b19e369874996d85e77bff20120e7b4d86c2d374caadcd
?> At what time (HH:MM) did the copy that worked start? :: sha256:5bb60732d06083e80e0c4e5c70219c0946c73e0d6b89cc8272e60c7015eff2c8
?> Which IP address received the data? :: sha256:992c86091ff8bac9b6fe49bd09a7fc9be514d3625aafcb290922153132ec0e6b

## 6. Check the indicators

Pull every indicator out of the evidence and check it against the intel feed and the client's allow list. Map what you saw to ATT&CK so the morning team and the client speak the same language.

- [ ] Extract indicators from the logs
- [ ] Which are on the feed, which are new, and which are on the allow list?
- [ ] An allowed destination received the stolen files. What does that mean for blocking?
- [ ] Map the behaviour to ATT&CK techniques
- [ ] Defang indicators before you paste them anywhere

> Hint: A phone call that talked the user into it, a remote access tool, three discovery commands, a download, a renamed program, and an upload to cloud storage.

```tool ioc
{"attack": ["T1566.004", "T1219", "T1033", "T1135", "T1083", "T1105", "T1036", "T1567.002"], "blank": ["iocs", "attack"]}
```

?> From which domain was the remote access program downloaded? :: sha256:75ac3e11ebbdb1c4fb575107d8fea2e979b89e2934cdde5d6156b73315d55f8b

## 7. Escalate

Write one escalation for the whole intrusion, not one per alert. The client's practice principal reads it on a phone, between clients.

- [ ] The ask first, with a time in UTC and in Brisbane time
- [ ] Severity and the fact that decided it
- [ ] Know / don't know
- [ ] What you have and haven't changed

> Hint: Say clearly that you have not touched the laptop. The client decides on isolation; you give them what they need to decide. Client tax files are personal information, so the client may have to tell people: say what was copied, not just that something was.

```tool report
{"tpl": "escalation", "title": "Escalation: client files leaving acc-lt03 through a remote access tool", "text": "# Escalation: client files leaving acc-lt03 through a remote access tool\n\n## The ask\n\nPlease approve isolating acc-lt03 (10.40.0.23) from the network now, by 01:30 UTC (11:30 Brisbane), and ask j.walsh to stop using it. Practice principal, please call the SOC on the bridge; please also ask j.walsh who phoned her this morning.\n\n## Severity and why\n\nHigh: about 2.2 GB of client tax files have already been uploaded to a cloud storage account the firm doesn't own, and someone outside has remote control of the laptop right now.\n\n## What we know\n\n- 00:03 UTC: j.walsh downloaded LedgerPro_Support.exe from ledgerpro-support.example, a domain registered three days ago that copies the name of the firm's real accounting software (AL-8401, fw_sessions.csv, dns.log).\n- 00:04 UTC: it is a remote access tool (RemoteHelper). It installed a service and connected to relay.remotehelper.example (AL-8403, System 7045, Sysmon).\n- 00:09 to 00:10 UTC: through the remote session, someone ran whoami, net use and a count of PDFs on S:\\Clients\\2026 (AL-8404).\n- 00:12 to 00:14 UTC: PowerShell downloaded a zip from downloads.toolmirror.example and unpacked svc-sync.exe into C:\\ProgramData\\Sync. Its real name is rclone.exe, a cloud copy tool (AL-8408, Sysmon event 1).\n- 00:15 UTC: an upload to filedrop.example was blocked by the firewall.\n- 00:16 UTC onwards: svc-sync.exe copied S:\\Clients\\2026 to storage.cloudbox.example (203.0.113.150) over 8 connections, about 2.2 GB (AL-8406, fw_sessions.csv, capture.pcap).\n\n## What we don't know\n\n- Exactly which client files were copied, and whether the copy has finished.\n- Who phoned j.walsh, and whether she gave them a password as well.\n- Whether the cloud storage account (wf-bkp-1112) can be reported to the storage provider for removal.\n\n## Impact so far\n\nOne laptop, acc-lt03, and one user, j.walsh. The copied folder holds 2026 tax files for the firm's clients: personal and financial information. Only acc-lt03 contacted the fake support site. Reception's normal uploads to the same storage site are small and expected.\n\n## What I've done\n\nNothing has been changed on the laptop. I have added ledgerpro-support.example, relay.remotehelper.example and downloads.toolmirror.example to the SIEM watch list.\n\n## Evidence\n\nalerts-night.json, fw_sessions.csv, dns.log, process.csv, the three event logs and capture.pcap are saved in the shift folder with SHA-256 hashes.\n", "n": 7, "blank": ["text"]}
```

## 8. Leave a new detection behind

The client's upload rule (AL-8406) fired at 00:46, thirty minutes after the copy started, because it waits for 1 GB. Write a better one: something the copy tool did **before** any data moved, that normal programs don't do. Test it, and write what would make it noisy.

- [ ] A Sigma rule on Sysmon process events
- [ ] Test it: it should match the renamed copy tool and nothing else
- [ ] When would it have fired, compared with AL-8406?
- [ ] One tuning note

> Hint: Sysmon records OriginalFileName from inside the program, so renaming the file doesn't change it. A program whose OriginalFileName is rclone.exe but whose file name is something else is hiding.

```tool sigma
{"rule": "title: Renamed rclone\nid: 5a1c7e10-0b8f-4a59-9a3e-2c1f00000402\nstatus: experimental\ndescription: rclone copies files to cloud storage. Running it under another name is how attackers hide data theft.\nlogsource:\n  product: windows\n  service: sysmon\ndetection:\n  selection:\n    EventID: 1\n    OriginalFileName: 'rclone.exe'\n  filter:\n    Image|endswith: '\\rclone.exe'\n  condition: selection and not filter\nfalsepositives:\n  - IT tools that bundle rclone under their own name (list them by path and signer)\nlevel: high\ntags:\n  - attack.exfiltration\n  - attack.t1567.002\n  - attack.defense_evasion\n  - attack.t1036\n", "n": 3, "ls": true, "blank": ["rule"]}
```

?> How many events does your renamed-rclone rule match? :: sha256:4e07408562bedb8b60ce05c1decfe3ad16b72230967de01f640b7e4729b49fce
?> How many minutes before AL-8406 would it have fired? :: sha256:e29c9c180c6279b0b02abd6a1801c7c04082cf486ec027aa13515e4f3884bb6b

## 9. Shift report

Close your shift. Everything open, by ID, with an owner. Everything closed in one line. What the morning should watch. Who you called and when. How many hints you opened.

- [ ] Open items by ID with owners
- [ ] Closed items
- [ ] Watch list including your new rule
- [ ] Calls with times
- [ ] Hints opened

```tool report
{"tpl": "handover", "title": "Shift report: night shift, 11 to 12 Nov", "text": "# Shift report: night shift, 11 to 12 Nov\n\n## Still open\n\n- INC-1112 (AL-8401, AL-8403, AL-8404, AL-8406, AL-8408): client files copied from acc-lt03 to cloud storage through a remote access tool. Escalated at 01:18 UTC. Next: client isolates the laptop; morning analyst lists exactly which files were copied and helps the client decide who must be told. Owner: morning shift lead.\n- AL-8412: unknown device android-7f3a on the staff Wi-Fi. Needs more information: ask the client's office manager whether a staff member connected a new phone today. Owner: morning analyst, by 03:00 UTC.\n\n## Closed this shift\n\n- AL-8402: benign, a.nguyen's own inbox rule for tax-office letters, made in Outlook from the office.\n- AL-8405: benign, Windows Update restarts during the patch push CHG-0417.\n- AL-8407: benign, typing mistakes with Caps Lock on, then a good logon from the same desk.\n- AL-8409: benign, the monthly archive job CHG-0412.\n- AL-8410: benign, service installed during the patch window CHG-0418.\n- AL-8411: benign, new phone, MFA approved, from the office's own address.\n\n## Watch for\n\n- Any host contacting ledgerpro-support.example, relay.remotehelper.example or downloads.toolmirror.example.\n- Any RemoteHelper service, or any program whose OriginalFileName is rclone.exe.\n- New rule \"Renamed rclone\" goes live tonight in test mode: check its hits.\n\n## Calls and messages\n\n- 01:18 UTC: called the client's practice principal; answered, approved isolation at 01:24 UTC.\n- 01:35 UTC: messaged the morning shift lead with the escalation; read at 01:37 UTC.\n\n## Tools and access\n\nWatch list entries for the three domains added at 01:10 UTC; review on 19 Nov. Hints opened: 0.\n", "n": 5, "blank": ["text"]}
```

## 10. Hand in

Hand in this notebook. It carries your queue, searches, escalation, rule and shift report.

**Rubric**

| Criterion | Points |
|---|---|
| Queue: all twelve alerts have a disposition and a reason backed by a log line | 6 |
| Scope: the one affected host, user, tool and destination named, with the evidence | 4 |
| Escalation: the ask first, severity with its reason, know / don't know, evidence location | 5 |
| New detection: a Sigma rule that catches the renamed copy tool before data moves, tested, with a tuning note | 4 |
| Shift report: open items by ID with owners, closed items, watch list, calls with times | 4 |
| Hints: minus 1 point for each hint opened (write the count in the shift report) | 0 |
| Working alone: every search and command kept in the notebook | 2 |

- [ ] All twelve alerts dispositioned
- [ ] Escalation and shift report show every section complete
- [ ] Rule tested
- [ ] Hints counted
- [ ] Press Hand in